# 03 — Segmentación de la cohorte de alto riesgo

El clustering no compite con el modelo predictivo: lo complementa. El modelo
dice *quién* tiene riesgo alto. El clustering pregunta si dentro de ese grupo
hay perfiles distintos que justifiquen acciones de retención distintas.

Dos decisiones de diseño:

**Se clusteriza sólo dentro de la cohorte priorizada**, no sobre toda la
cartera. Agrupar los 2.026 clientes del holdout produciría sobre todo la
separación entre activos e inactivos, que ya conocemos por el modelo y que no
agrega nada accionable.

**Se reduce la dimensión con PCA antes de aplicar k-means.** K-means mide
distancia euclídea, que en alta dimensión se concentra y pierde poder de
discriminación, y las dummies del one-hot inflan la dimensión aportando poca
varianza.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src import config as C
from src import data as D
from src import evaluate as E
from src import interpret as I
from src import segment as S
from src.features import add_engineered_features, build_preprocessor
from src.models import build_model_zoo

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 40)

In [ ]:
from sklearn.model_selection import StratifiedKFold

train = add_engineered_features(D.load_split("train"))
test = add_engineered_features(D.load_split("test"))
X_train, y_train = D.split_X_y(train)
X_test, y_test = D.split_X_y(test)

# Se carga el modelo ya ajustado y el umbral congelado que dejó
# scripts/run_pipeline.py. Reajustar acá con hiperparámetros por defecto daría
# un modelo distinto del que se validó, y por lo tanto otra cohorte.
import joblib
import json

congelado = json.loads((C.REPORTS / "results" / "modelo_congelado.json").read_text())
umbral = congelado["umbral"]
mejor = joblib.load(C.MODELS_DIR / "modelo_final.joblib")

proba_test = mejor.predict_proba(X_test)[:, 1]

cohorte = test.copy()
cohorte["score"] = proba_test
cohorte = cohorte[cohorte["score"] >= umbral].copy()

print(f"Cohorte priorizada: {len(cohorte)} clientes "
      f"({len(cohorte) / len(test):.1%} del holdout)")
print(f"Tasa de abandono en la cohorte: {cohorte[C.TARGET].mean():.3f} "
      f"(vs {test[C.TARGET].mean():.3f} en la cartera completa)")

## 1. Cuántos grupos

Se comparan silhouette, Calinski-Harabasz e inercia. La inercia decrece siempre
al aumentar K, así que por sí sola no decide nada; el codo suele ser ambiguo con
datos de clientes.

In [ ]:
Z, _ = S.build_cluster_matrix(cohorte, mejor, n_components=0.90)
print(f"Matriz de clustering: {Z.shape[0]} clientes × {Z.shape[1]} componentes "
      f"(90% de la varianza)")

k, diagnostico = S.choose_k(Z)
diagnostico

El máximo de silhouette está en K=2 (0.185) y cae abruptamente a 0.099
en K=3. Calinski-Harabasz apunta en la misma dirección.

Hay que ser honesto sobre la magnitud: un silhouette de 0.185 es **bajo**.
Indica que los grupos se solapan considerablemente. Con datos de comportamiento
de clientes esto es lo esperable —no hay tipos discretos de cliente, hay un
continuo— pero significa que la segmentación describe una tendencia, no una
partición nítida. Presentarla como "encontramos dos perfiles claramente
diferenciados" sería exagerar lo que los datos sostienen.

In [ ]:
labels = S.fit_kmeans(Z, k)

# contraste con un modelo de mezclas gaussianas, que admite clusters
# elípticos y de tamaños distintos
labels_gmm = S.fit_gmm(Z, k)
acuerdo = (labels == labels_gmm).mean()
acuerdo = max(acuerdo, 1 - acuerdo)  # las etiquetas son arbitrarias
print(f"Acuerdo entre k-means y GMM: {acuerdo:.1%}")

## 2. Perfil de los segmentos

En unidades originales, no estandarizadas: el equipo comercial necesita leer
"transaccionan 41 veces al año", no "-0,8 desvíos".

In [ ]:
columnas = ["Total_Trans_Ct", "Total_Trans_Amt", "ticket_promedio",
            "Credit_Limit", "Total_Revolving_Bal", "Total_Relationship_Count",
            "Contacts_Count_12_mon", "Total_Ct_Chng_Q4_Q1", "Customer_Age"]

perfil = S.profile_clusters(cohorte, labels, columnas)
perfil.columns = [C.label(c) if c in C.LABELS_ES else c for c in perfil.columns]
perfil

Los dos segmentos, sobre la cohorte priorizada del holdout:

**Segmento 0 — alto volumen, 20% de la cohorte (79 clientes).** Límite de
tarjeta de ~14.500, monto anual de ~7.500 con ticket promedio de ~109, y 69
transacciones al año. Su indicador de deterioro es la desaceleración: la
variación trimestral cayó a 0,80.

**Segmento 1 — bajo volumen, 80% de la cohorte (317 clientes).** Límite de
~6.900, monto anual de ~2.200 con ticket promedio de ~53, y 41 transacciones.
La caída trimestral es mucho más pronunciada (0,51) y el ratio de utilización
es tres veces mayor.

Las tasas de abandono son casi idénticas (0,80 y 0,77): **los segmentos no se
diferencian en riesgo, se diferencian en valor**. Esa es exactamente su
utilidad. El modelo ya ordenó por riesgo; el clustering agrega la dimensión que
falta para decidir cuánto invertir en retener a cada uno.

In [ ]:
cat = S.categorical_profile(cohorte, labels)
cat

La composición demográfica de los dos segmentos es prácticamente la
misma, consistente con el notebook 01: la demografía no separa nada en este
dataset. La segmentación es puramente conductual.

## 3. Qué hacer con esto

La segmentación sugiere dos tratamientos distintos, con la salvedad de que el
solapamiento es alto y esto es una orientación, no una asignación estricta:

**Segmento 0 (alto volumen).** Pocos clientes, mucho valor por cliente.
Justifica contacto humano y ofertas individualizadas. La señal a monitorear es
la desaceleración trimestral, que aparece antes de que caiga el volumen anual.

**Segmento 1 (bajo volumen).** Cuatro veces más clientes, un tercio del monto
por cliente. El costo de contacto individual no se paga; corresponden acciones
automatizadas. La utilización de crédito alta con montos bajos sugiere clientes
con límite ajustado, para quienes una revisión de límite puede ser más efectiva
que un descuento.

**Lo que estos datos no permiten decir.** No hay costo de adquisición, margen
por cliente ni valor de vida, así que no se puede calcular el retorno de una
campaña. Los costos usados para fijar el umbral (100 y 15) son supuestos
ilustrativos elegidos por su relación, no cifras de un banco. Cualquier
implementación real tiene que reemplazarlos por números propios antes de
convertir estas probabilidades en presupuesto.